# 从行为克隆搭出一台 Tiny VLA

VLA 的核心输出是动作。我们先用机器人状态模仿专家，再加入图片、文字和 action chunk。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'hwm').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import torch
from hwm.robot import (
    TinyVLA, INSTRUCTIONS, evaluate_vla, make_tabletop_dataset,
)
torch.manual_seed(0)


## 1. 一条机器人示范里有什么

同一时刻需要对齐图片、语言指令、机器人自身状态和动作。这里一次保存未来三个动作。

In [ ]:
data = make_tabletop_dataset(num_samples=160, chunk_size=3, seed=0)
for name, value in data.items(): print(f'{name:14s}', tuple(value.shape))
print('第一条指令:', INSTRUCTIONS[int(data['instructions'][0])])
assert data['action_chunks'].shape == (160, 3, 2)


## 2. 最小 state-only 行为克隆

先不使用图像和文字，只检查监督学习能否从 state 预测专家第一步。简单基线能帮助我们判断视觉模型是否真的增加价值。

In [ ]:
state_policy = torch.nn.Sequential(torch.nn.Linear(8 + 2, 32), torch.nn.ReLU(), torch.nn.Linear(32, 2), torch.nn.Tanh())
instruction_onehot = torch.nn.functional.one_hot(data['instructions'], 2).float()
state_input = torch.cat((data['states'], instruction_onehot), dim=-1)
target = data['action_chunks'][:, 0]
opt = torch.optim.Adam(state_policy.parameters(), lr=3e-3)
losses = []
for _ in range(50):
    opt.zero_grad(); prediction = state_policy(state_input); loss = torch.nn.functional.mse_loss(prediction, target); loss.backward(); opt.step(); losses.append(float(loss.detach()))
print('state BC loss:', round(losses[0], 3), '→', round(losses[-1], 3))
assert losses[-1] < losses[0]


## 3. 加入图像与语言，输出 action chunk

CNN 读取桌面图片，language embedding 区分红色与绿色目标，proprioception 告诉模型抓手精确位置。三个动作一次输出。

In [ ]:
model = TinyVLA(chunk_size=3)
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
losses = []
for _ in range(60):
    opt.zero_grad(); chunks = model(data['images'], data['instructions'], data['states']); loss = torch.nn.functional.mse_loss(chunks, data['action_chunks']); loss.backward(); opt.step(); losses.append(float(loss.detach()))
print('multimodal chunk loss:', round(losses[0], 3), '→', round(losses[-1], 3))
print('output:', tuple(chunks.shape))
assert losses[-1] < losses[0]


## 4. 同一场景换指令

如果文字真的参与决策，同一张图从‘去红色’改成‘去绿色’，动作应该改变。

In [ ]:
same_image = data['images'][:1].expand(2, -1, -1, -1)
same_state = data['states'][:1].expand(2, -1)
with torch.no_grad(): two_goals = model(same_image, torch.tensor([0, 1]), same_state)
difference = (two_goals[0] - two_goals[1]).abs().mean()
print('换目标后的动作差异:', round(float(difference), 4))
assert difference > 0


## 5. 回到环境连续执行

监督 loss 下降以后，我们让模型每一步重新看图并执行 action chunk 的第一步。成功率、碰撞和最终距离会揭示动作 MSE 没有显示的问题。

In [ ]:
test_data = make_tabletop_dataset(32, chunk_size=3, seed=17)
metrics = evaluate_vla(
    model, test_data['states'], test_data['instructions'], max_steps=12
)
for name in ('success_rate', 'mean_collisions',
             'initial_distance', 'final_distance'):
    print(name, round(metrics[name], 3))
print('若 loss 下降而成功率仍低，模型只是拟合了单步示范，'
      '没有解决闭环分布偏移。')
assert 0 <= metrics['success_rate'] <= 1


## 小结

这一份的模型已经能从图像、指令与状态输出动作，但它没有预测动作后果。动作 MSE 下降也不保证闭环成功；本实验会诚实保留这种差距。第二份 Notebook 会让模型先检查候选动作。